# 03 - Transfer Governance Before Export

Destination-aware authorization: the SAME asset and operation produce different
typed answers per destination and consumer jurisdiction, because the server
evaluates the authored transfer rules at read time.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None):
    ref = {"database": "acmecloud_demo", "schema": "public", "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## Salesforce (US) for EU consumers → conditional, with typed conditions


In [ ]:
salesforce = client.authorize_use(
    asset("customers"),
    use="sync approved customer fields to the CRM",
    scenario_key="residency.cross_border_transfer",
    operation="export",
    destination={"system": "SALESFORCE", "jurisdiction": "US"},
    consumer_jurisdiction="EU",
)
print_answer(salesforce)


## Advertising platform → deny · External LLM vendor → deny


In [ ]:
ads = client.authorize_use(
    asset("customers"),
    use="send the customer batch to the advertising platform",
    scenario_key="residency.cross_border_transfer",
    operation="export",
    destination={"system": "ADS_PLATFORM", "jurisdiction": "US"},
    consumer_jurisdiction="US",
)
llm = client.authorize_use(
    asset("customers"),
    use="send the customer batch to an external LLM vendor",
    scenario_key="residency.cross_border_transfer",
    operation="export",
    destination={"system": "EXTERNAL_LLM_VENDOR", "jurisdiction": "US"},
    consumer_jurisdiction="US",
)
print(f"ADS_PLATFORM        -> {ads['decision']}")
print(f"EXTERNAL_LLM_VENDOR -> {llm['decision']}")


## An unmatched destination falls back to the authored default

No rule names `INTERNAL_WAREHOUSE`, so the policy's `defaultEffect`
(conditional) answers — nothing is silently allowed.


In [ ]:
unmatched = client.authorize_use(
    asset("customer_exports"),
    use="stage the export batch in the internal warehouse",
    scenario_key="residency.cross_border_transfer",
    operation="export",
    destination={"system": "INTERNAL_WAREHOUSE", "jurisdiction": "US"},
    consumer_jurisdiction="US",
)
print_answer(unmatched)


## Chain the conditional decision into `explain_why`


In [ ]:
explanation = client.explain_why(salesforce["decision_id"])
print(f"current: {explanation['current']}")
print(explanation["explanation"])
